In [ ]:
#@title 🎮 第1歩：まずはファミコンゲームを体験しよう！ { display-mode: "form" }
#@markdown 再生ボタン（▶）を押すと、自動的にC言語のプログラムがコンパイルされ、下にゲーム画面が現れます。
#@markdown #@markdown 画面が表示されたら、**画面内を一度クリック**して音声をONにし、キーボードで操作してみましょう！


# 0. ターゲット設定
# ────【ここを書き換えて課題を切り替える（拡張子は不要）】────
TARGET = "game"
# ────────────────────────────────────────────────────────


# 1. 開発環境（cc65コンパイラ）のインストール（全員共通）
!apt-get install cc65


# 2. 【オプション】Googleドライブを接続するための関数定義 # （学校のアカウントなどでエラーが出る場合は、一番下の「mount_drive()」の先頭に「#」をつけて飛ばしてください）
import os
def mountdrive():
  from google.colab import drive
  drive.mount('/content/drive')
  target_dir = "/content/drive/MyDrive"
  os.makedirs(target_dir, exist_ok=True)
  %cd $target_dir
# ─── ドライブを使いたい人だけ、下の行の「#」を消して実行してください ───
# mountdrive()


# 3. 初回のみ、環境（NESLab）を各自のドライブにダウンロードして解凍
# （すでにフォルダが存在する場合はスキップする処理を入れると親切です）
# ドライブ未接続なら /content に戻り、接続済なら直前の%cdを維持する対策
if 'target_dir' not in locals():
 %cd /content
if not os.path.exists("NESLab"):
 !git clone https://github.com/ip-arch/NESLab.git
# 4. 確実に「正しい階層の」NESLabに移動する


if os.path.exists("NESLab"):
 %cd NESLab
 !pwd
!ls
# 5. game.nes作成


import os
import base64
from IPython.display import HTML


c_file = f"{TARGET}.c"
nes_file = f"{TARGET}.nes"


print(f"現在地: {os.getcwd()}")


# 1. 指定されたCファイルが存在するかチェック
if not os.path.exists(c_file):
    print(f"❌ エラー: {c_file} が見つかりません。ファイル名を確認してください。")
else:
    # 2. 古い成果物を削除して、指定された課題ファイルのみをビルド
    print(f"📦 {c_file} をコンパイル中...")
    !make clean
    !make {nes_file} M65CDIR=/usr


    # 3. ビルドに成功したら、そのNESファイルを読み込んでエミュレータを起動
    if os.path.exists(nes_file):
        print(f"🚀 {nes_file} の起動に成功しました！下の画面をクリックして音声を有効にしてください。")

        with open(nes_file, "rb") as f:
            nes_bytes = f.read()
        nes_base64 = base64.b64encode(nes_bytes).decode('utf-8')


        # 音声対応版エミュレータHTML（読み込むファイルを動的に変更）
        html_code = f"""
        <div id="nes-container" style="text-align: center; background: #222; padding: 15px; border-radius: 8px; color: white; font-family: sans-serif; max-width: 540px; margin: 0 auto; cursor: pointer;">
            <h4 style="margin-top: 0; color: #ff4a4a;">🔊 NES Emulator [{nes_file}]</h4>
            <canvas id="nes-canvas" width="256" height="240" style="width: 512px; height: 480px; background: black; border: 4px solid #444; border-radius: 4px;"></canvas>
            <p style="font-size: 0.85rem; color: #ccc; margin-top: 10px; line-height: 1.4;">
                【重要】 <b>画面内を一度クリックするとサウンドが有効になります！</b><br>
                <b>十字キー</b>: 移動 | <b>Zキー</b>: Bボタン | <b>Xキー</b>: Aボタン<br>
                <b>Enterキー</b>: START | <b>Spaceキー</b>: SELECT
            </p>
        </div>


        <script src="https://cdnjs.cloudflare.com/ajax/libs/jsnes/1.2.1/jsnes.min.js"></script>
        <script>
        (function() {{
            var canvas = document.getElementById('nes-canvas');
            var ctx = canvas.getContext('2d');
            var imageData = ctx.getImageData(0, 0, 256, 240);


            var AUDIO_BUFFER_SIZE = 1024;
            var SAMPLE_COUNT = 44100 * 2;
            var audioBuffer = new Float32Array(SAMPLE_COUNT);
            var bufferReadIdx = 0;
            var bufferWriteIdx = 0;
            var audioCtx = null;
            var scriptNode = null;


            function initAudio() {{
                if (audioCtx) return;
                audioCtx = new (window.AudioContext || window.webkitAudioContext)({{ sampleRate: 44100 }});
                scriptNode = audioCtx.createScriptProcessor(AUDIO_BUFFER_SIZE, 0, 2);

                scriptNode.onaudioprocess = function(e) {{
                    var outputLeft = e.outputBuffer.getChannelData(0);
                    var outputRight = e.outputBuffer.getChannelData(1);
                    var available = (bufferWriteIdx - bufferReadIdx + SAMPLE_COUNT) % SAMPLE_COUNT;

                    if (available > 8192) {{
                        bufferReadIdx = (bufferWriteIdx - 4096 + SAMPLE_COUNT) % SAMPLE_COUNT;
                    }}


                    for (var i = 0; i < AUDIO_BUFFER_SIZE; i++) {{
                        if (bufferReadIdx === bufferWriteIdx) {{
                            outputLeft[i] = 0; outputRight[i] = 0;
                        }} else {{
                            outputLeft[i] = audioBuffer[bufferReadIdx];
                            bufferReadIdx = (bufferReadIdx + 1) % SAMPLE_COUNT;
                            outputRight[i] = audioBuffer[bufferReadIdx];
                            bufferReadIdx = (bufferReadIdx + 1) % SAMPLE_COUNT;
                        }}
                    }}
                }};
                scriptNode.connect(audioCtx.destination);
            }}


            var nes = new jsnes.NES({{
                onFrame: function(frameBuffer) {{
                    var d = imageData.data;
                    for (var i = 0; i < frameBuffer.length; i++) {{
                        var p = frameBuffer[i];
                        var idx = i * 4;
                        d[idx]     = (p >> 16) & 0xff;
                        d[idx + 1] = (p >> 8) & 0xff;
                        d[idx + 2] = p & 0xff;
                        d[idx + 3] = 0xff;
                    }}
                    ctx.putImageData(imageData, 0, 0);
                }},
                onAudioSample: function(left, right) {{
                    if (!audioCtx) return;
                    audioBuffer[bufferWriteIdx] = left;
                    bufferWriteIdx = (bufferWriteIdx + 1) % SAMPLE_COUNT;
                    audioBuffer[bufferWriteIdx] = right;
                    bufferWriteIdx = (bufferWriteIdx + 1) % SAMPLE_COUNT;
                }}
            }});


            var romData = atob("{nes_base64}");
            nes.loadROM(romData);


            var keyboard = function(callback, event) {{
                var player = 1;
                switch(event.keyCode) {{
                    case 38: callback(player, jsnes.Controller.BUTTON_UP); event.preventDefault(); break;
                    case 40: callback(player, jsnes.Controller.BUTTON_DOWN); event.preventDefault(); break;
                    case 37: callback(player, jsnes.Controller.BUTTON_LEFT); event.preventDefault(); break;
                    case 39: callback(player, jsnes.Controller.BUTTON_RIGHT); event.preventDefault(); break;
                    case 88: callback(player, jsnes.Controller.BUTTON_A); event.preventDefault(); break;
                    case 90: callback(player, jsnes.Controller.BUTTON_B); event.preventDefault(); break;
                    case 13: callback(player, jsnes.Controller.BUTTON_START); event.preventDefault(); break;
                    case 32: callback(player, jsnes.Controller.BUTTON_SELECT); event.preventDefault(); break;
                }}
            }};


            document.addEventListener('keydown', function(e) {{ keyboard(nes.buttonDown, e); }});
            document.addEventListener('keyup', function(e) {{ keyboard(nes.buttonUp, e); }});


            document.getElementById('nes-container').addEventListener('click', function() {{
                initAudio();
                if (audioCtx && audioCtx.state === 'suspended') {{ audioCtx.resume(); }}
                document.querySelector('#nes-container h4').style.color = '#4ae2ff';
                document.querySelector('#nes-container h4').innerText = '🎮 NES Emulator [{nes_file}] (Sound ACTIVE)';
            }});


            function step() {{ nes.frame(); requestAnimationFrame(step); }}
            requestAnimationFrame(step);
        }})();
        </script>
        """
        display(HTML(html_code))
    else:
        print(f"❌ エラー: {nes_file} の生成に失敗しました。Cコードの構文などを確認してください。")







